In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score
from src.preprocessing import train_valid_test_split
from src.evaluate import precision_k, score_precision_10
from sklearn.model_selection import GridSearchCV

In [2]:
df = pd.read_csv('../data/bank-full.csv', sep=';')
train, valid, test = train_valid_test_split(df)
k = int(len(valid) * 0.1)

In [3]:
train['pdays_prev_camp'] = train['pdays'].ne(-1).astype('int8')
train['pdays'] = train['pdays'].replace(-1, np.nan)

valid['pdays_prev_camp'] = valid['pdays'].ne(-1).astype('int8')
valid['pdays'] = valid['pdays'].replace(-1, np.nan)

/var/folders/d5/hmchxs4s0v7bc772rvxtlcdw0000gn/T/ipykernel_99557/759346521.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['pdays_prev_camp'] = train['pdays'].ne(-1).astype('int8')
/var/folders/d5/hmchxs4s0v7bc772rvxtlcdw0000gn/T/ipykernel_99557/759346521.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['pdays'] = train['pdays'].replace(-1, np.nan)
/var/folders/d5/hmchxs4s0v7bc772rvxtlcdw0000gn/T/ipykernel_99557/759346521.py:4: SettingWithCopyWarning: 
A value is trying to be set on a cop

In [4]:
X_train = train.drop(columns=['y', 'duration'])
y_train = train["y"]
X_valid = valid.drop(columns=['y', 'duration'])
y_valid = valid['y']

In [5]:

cat_cols = X_train.select_dtypes(
    include=["object", "category", "string", "bool"]
)
cat_indices = [
      i for i, col in enumerate(X_train.columns)
      if X_train[col].dtype.name in ("object", "string", "category")
        ]

# eval_pool = Pool(X_valid, y_valid, cat_features=cat_indices)

In [6]:
param_grid = {
      "depth": [4, 6, 8],
      "learning_rate": [0.01, 0.03, 0.1],
      "l2_leaf_reg": [3, 10],
}


In [13]:
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.01,
    loss_function="Logloss",
    depth=8,
    random_seed=0,
    l2_leaf_reg=3,
)
model.fit(X_train, y_train, cat_features=cat_indices, verbose=100)

0:	learn: 0.6807804	total: 55ms	remaining: 55s
1:	learn: 0.6687512	total: 89.2ms	remaining: 44.5s
2:	learn: 0.6568578	total: 115ms	remaining: 38.4s
3:	learn: 0.6454265	total: 145ms	remaining: 36.2s
4:	learn: 0.6344082	total: 171ms	remaining: 33.9s
5:	learn: 0.6235711	total: 190ms	remaining: 31.5s
6:	learn: 0.6131413	total: 203ms	remaining: 28.8s
7:	learn: 0.6029956	total: 216ms	remaining: 26.8s
8:	learn: 0.5930780	total: 235ms	remaining: 25.9s
9:	learn: 0.5833989	total: 288ms	remaining: 28.5s
10:	learn: 0.5738939	total: 359ms	remaining: 32.2s
11:	learn: 0.5647503	total: 384ms	remaining: 31.6s
12:	learn: 0.5558257	total: 412ms	remaining: 31.3s
13:	learn: 0.5469155	total: 471ms	remaining: 33.2s
14:	learn: 0.5384486	total: 516ms	remaining: 33.9s
15:	learn: 0.5301210	total: 572ms	remaining: 35.2s
16:	learn: 0.5220121	total: 611ms	remaining: 35.3s
17:	learn: 0.5142603	total: 628ms	remaining: 34.3s
18:	learn: 0.5066164	total: 655ms	remaining: 33.8s
19:	learn: 0.4992002	total: 696ms	remaining

CatBoostClassifier(depth=8, iterations=1000, l2_leaf_reg=3, learning_rate=0.01, loss_function='Logloss', random_seed=0)

In [11]:
# X_search = pd.concat([X_train, X_valid], ignore_index=True)
# y_search = pd.concat([y_train, y_valid], ignore_index=True)
#
#
# grid = GridSearchCV(estimator=model,
#     param_grid=param_grid,
#     cv=[(np.arange(len(X_train)), np.arange(len(X_train), len(X_search)))],
#     scoring=score_precision_10,
#     refit=False,
#     n_jobs=1,
#     error_score='raise',
# )
#
# grid.fit(X_search, y_search, cat_features=cat_indices, verbose=100)


0:	learn: 0.6808320	total: 78ms	remaining: 1m 17s
100:	learn: 0.2537676	total: 2.53s	remaining: 22.5s
200:	learn: 0.2136609	total: 4.5s	remaining: 17.9s
300:	learn: 0.2067578	total: 6.77s	remaining: 15.7s
400:	learn: 0.2044245	total: 8.63s	remaining: 12.9s
500:	learn: 0.2032859	total: 10.5s	remaining: 10.4s
600:	learn: 0.2025208	total: 13.1s	remaining: 8.67s
700:	learn: 0.2018126	total: 15.5s	remaining: 6.61s
800:	learn: 0.2013086	total: 18.2s	remaining: 4.53s
900:	learn: 0.2008055	total: 20.6s	remaining: 2.26s
999:	learn: 0.2002006	total: 22.7s	remaining: 0us
0:	learn: 0.6567930	total: 19ms	remaining: 19s
100:	learn: 0.2068929	total: 1.99s	remaining: 17.7s
200:	learn: 0.2025989	total: 4.03s	remaining: 16s
300:	learn: 0.2008391	total: 6.04s	remaining: 14s
400:	learn: 0.1991549	total: 7.95s	remaining: 11.9s
500:	learn: 0.1974383	total: 9.85s	remaining: 9.81s
600:	learn: 0.1961624	total: 11.9s	remaining: 7.89s
700:	learn: 0.1950398	total: 14.1s	remaining: 6.03s
800:	learn: 0.1941744	tota

GridSearchCV(cv=[(array([    0,     1,     2, ..., 31644, 31645, 31646]),
                  array([31647, 31648, 31649, ..., 38426, 38427, 38428]))],
             error_score='raise',
             estimator=CatBoostClassifier(depth=6, iterations=1000, learning_rate=0.01, loss_function='Logloss', random_seed=0),
             n_jobs=1,
             param_grid={'depth': [4, 6, 8], 'l2_leaf_reg': [3, 10],
                         'learning_rate': [0.01, 0.03, 0.1]},
             refit=False, scoring=<function score_precision_10 at 0x1683805e0>)

In [12]:
# grid.best_params_

{'depth': 8, 'l2_leaf_reg': 3, 'learning_rate': 0.01}

In [14]:
y_pred = model.predict(X_valid)

In [15]:
yes_idx = list(model.classes_).index("yes")
y_score = model.predict_proba(X_valid)[:, yes_idx]
prec_k = precision_k(y_valid, y_score, k=k)
avg_precision = average_precision_score(y_valid, y_score, pos_label="yes")
# avg_precision = model.eval_metrics(data=eval_pool, metrics='Precision')

In [16]:
pd.Series({
    "precision_at_10pct": prec_k,
    "average_precision": avg_precision,
})

precision_at_10pct    0.297935
average_precision     0.238877
dtype: float64

#### exp 1:
- depth 6, iteration 500, random seed = 0, logloss, lr = 0.01
- metrics: prec_k = 0.2743, avg_prec = 0.2322

#### exp 2:
- same model pars
- metrics: prec_k = 0.2935, avg_prec = 0.2361

#### exp 3:
- {'depth': 8, 'l2_leaf_reg': 3, 'learning_rate': 0.01}
- metrics: prec_k = 0.2979, avg_prec = 0.2388